In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report
)

In [2]:
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

OUTPUT_DIR = Path("mmi711_outputs_multilabel")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Original event-presence dataset
data = np.load(OUTPUT_DIR / "mmi711_multilabel_variable_length_datasets.npz")

# New sequence-location labels
loc_data = np.load(OUTPUT_DIR / "mmi711_sequence_location_labels.npz")

with open(OUTPUT_DIR / "dataset_config_multilabel_variable_lengths.json", "r") as f:
    config = json.load(f)

LENGTHS = config["lengths"]
EVENT_LABELS = config["event_labels"]

print("Lengths:", LENGTHS)
print("Event labels:", EVENT_LABELS)
print("Location label arrays:", loc_data.files[:5], "...")

Device: cuda
Lengths: [400, 800, 1200]
Event labels: ['mean_shift', 'variance_shift', 'trend_shift', 'point_anomaly', 'collective_anomaly']
Location label arrays: ['Yloc_train_L400', 'Yloc_train_L800', 'Yloc_train_L1200', 'Yloc_val_L400', 'Yloc_val_L800'] ...


In [3]:
X_train_by_length = {}
Yloc_train_by_length = {}

X_val_by_length = {}
Yloc_val_by_length = {}

X_ood_params_by_length = {}
Yloc_ood_params_by_length = {}

X_ood_background_by_length = {}
Yloc_ood_background_by_length = {}

for length in LENGTHS:
    X_train_by_length[length] = data[f"X_train_L{length}"]
    Yloc_train_by_length[length] = loc_data[f"Yloc_train_L{length}"]

    X_val_by_length[length] = data[f"X_val_L{length}"]
    Yloc_val_by_length[length] = loc_data[f"Yloc_val_L{length}"]

    X_ood_params_by_length[length] = data[f"X_ood_params_L{length}"]
    Yloc_ood_params_by_length[length] = loc_data[f"Yloc_ood_params_L{length}"]

    X_ood_background_by_length[length] = data[f"X_ood_background_L{length}"]
    Yloc_ood_background_by_length[length] = loc_data[f"Yloc_ood_background_L{length}"]

    print(
        f"Length {length}:",
        "train", X_train_by_length[length].shape, Yloc_train_by_length[length].shape,
        "| val", X_val_by_length[length].shape, Yloc_val_by_length[length].shape,
        "| ood_params", X_ood_params_by_length[length].shape, Yloc_ood_params_by_length[length].shape,
        "| ood_background", X_ood_background_by_length[length].shape, Yloc_ood_background_by_length[length].shape
    )

Length 400: train (800, 400) (800, 400, 5) | val (240, 400) (240, 400, 5) | ood_params (240, 400) (240, 400, 5) | ood_background (240, 400) (240, 400, 5)
Length 800: train (800, 800) (800, 800, 5) | val (240, 800) (240, 800, 5) | ood_params (240, 800) (240, 800, 5) | ood_background (240, 800) (240, 800, 5)
Length 1200: train (800, 1200) (800, 1200, 5) | val (240, 1200) (240, 1200, 5) | ood_params (240, 1200) (240, 1200, 5) | ood_background (240, 1200) (240, 1200, 5)


In [4]:
def check_location_labels(Yloc_by_length, split_name):
    print("\n" + "=" * 70)
    print(split_name)
    print("=" * 70)

    Y_all = np.concatenate(
        [Yloc_by_length[length].reshape(-1, len(EVENT_LABELS)) for length in LENGTHS],
        axis=0
    )

    for i, label in enumerate(EVENT_LABELS):
        values, counts = np.unique(Y_all[:, i], return_counts=True)
        print(label, dict(zip(values.tolist(), counts.tolist())))

check_location_labels(Yloc_train_by_length, "Train")
check_location_labels(Yloc_val_by_length, "Validation")


Train
mean_shift {0: 1586976, 1: 281803, 2: 51221}
variance_shift {0: 1588512, 1: 284205, 2: 47283}
trend_shift {0: 1598427, 1: 272749, 2: 48824}
point_anomaly {0: 1919626, 1: 374}
collective_anomaly {0: 1864539, 1: 55461}

Validation
mean_shift {0: 472606, 1: 87927, 2: 15467}
variance_shift {0: 473180, 1: 86763, 2: 16057}
trend_shift {0: 478320, 1: 84475, 2: 13205}
point_anomaly {0: 575889, 1: 111}
collective_anomaly {0: 559516, 1: 16484}


In [5]:
class TimeSeriesLocationDataset(Dataset):
    def __init__(self, X, Yloc):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Yloc = torch.tensor(Yloc, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Yloc[idx]

In [6]:
class CNNBiLSTMLocationModel(nn.Module):
    def __init__(self, num_shift_types=3, num_anomaly_types=2, shift_classes=3, hidden_size=64, dropout=0.2):
        super().__init__()

        self.num_shift_types = num_shift_types
        self.num_anomaly_types = num_anomaly_types
        self.shift_classes = shift_classes

        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.bilstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        feature_dim = hidden_size * 2

        self.dropout = nn.Dropout(dropout)

        # For mean/variance/trend shift:
        # output shape will be (batch, length, 3 shift types, shift_classes)
        self.shift_head = nn.Linear(feature_dim, num_shift_types * shift_classes)

        # For point/collective anomaly:
        # output shape will be (batch, length, 2 anomaly types)
        self.anomaly_head = nn.Linear(feature_dim, num_anomaly_types)

    def forward(self, x):
        # x: (batch, length)

        x = x.unsqueeze(1)              # (batch, 1, length)
        conv_features = self.cnn(x)     # (batch, 64, length)
        conv_features = conv_features.transpose(1, 2)  # (batch, length, 64)

        lstm_out, _ = self.bilstm(conv_features)       # (batch, length, hidden*2)
        features = self.dropout(lstm_out)

        shift_logits = self.shift_head(features)
        shift_logits = shift_logits.view(
            x.size(0),
            features.size(1),
            self.num_shift_types,
            self.shift_classes
        )

        anomaly_logits = self.anomaly_head(features)

        return shift_logits, anomaly_logits

In [7]:
BATCH_SIZE = 32

def create_location_loaders(X_by_length, Yloc_by_length, shuffle):
    loaders = []

    for length in LENGTHS:
        loader = DataLoader(
            TimeSeriesLocationDataset(
                X_by_length[length],
                Yloc_by_length[length]
            ),
            batch_size=BATCH_SIZE,
            shuffle=shuffle
        )
        loaders.append(loader)

    return loaders

train_loaders = create_location_loaders(X_train_by_length, Yloc_train_by_length, shuffle=True)
val_loaders = create_location_loaders(X_val_by_length, Yloc_val_by_length, shuffle=False)
ood_params_loaders = create_location_loaders(X_ood_params_by_length, Yloc_ood_params_by_length, shuffle=False)
ood_background_loaders = create_location_loaders(X_ood_background_by_length, Yloc_ood_background_by_length, shuffle=False)

print("Number of train loaders:", len(train_loaders))

Number of train loaders: 3


In [8]:
max_shift_label = 0

for length in LENGTHS:
    max_shift_label = max(
        max_shift_label,
        int(Yloc_train_by_length[length][:, :, :3].max()),
        int(Yloc_val_by_length[length][:, :, :3].max())
    )

SHIFT_CLASSES = max_shift_label + 1

print("Max shift label:", max_shift_label)
print("Number of shift classes:", SHIFT_CLASSES)

Max shift label: 2
Number of shift classes: 3


In [9]:
def compute_anomaly_pos_weight(Yloc_by_length):
    all_anom = []

    for length in LENGTHS:
        # point_anomaly and collective_anomaly columns
        all_anom.append(Yloc_by_length[length][:, :, 3:5].reshape(-1, 2))

    all_anom = np.concatenate(all_anom, axis=0)

    pos = all_anom.sum(axis=0)
    neg = len(all_anom) - pos

    pos_weight = neg / (pos + 1e-8)
    pos_weight = np.clip(pos_weight, 1.0, 50.0)

    return torch.tensor(pos_weight, dtype=torch.float32)


anomaly_pos_weight = compute_anomaly_pos_weight(Yloc_train_by_length).to(device)
print("Anomaly pos_weight:", anomaly_pos_weight)

Anomaly pos_weight: tensor([50.0000, 33.6189], device='cuda:0')


In [10]:
def compute_location_loss(
    shift_logits,
    anomaly_logits,
    Yloc,
    anomaly_pos_weight,
    shift_loss_weight=1.0,
    anomaly_loss_weight=1.0
):
    """
    shift_logits:   (batch, length, 3, shift_classes)
    anomaly_logits: (batch, length, 2)
    Yloc:           (batch, length, 5)
    """

    shift_targets = Yloc[:, :, :3].long()
    anomaly_targets = Yloc[:, :, 3:5].float()

    shift_losses = []

    for shift_idx in range(3):
        logits_i = shift_logits[:, :, shift_idx, :]      # (batch, length, classes)
        target_i = shift_targets[:, :, shift_idx]        # (batch, length)

        loss_i = F.cross_entropy(
            logits_i.reshape(-1, logits_i.size(-1)),
            target_i.reshape(-1)
        )
        shift_losses.append(loss_i)

    shift_loss = torch.stack(shift_losses).mean()

    anomaly_loss = F.binary_cross_entropy_with_logits(
        anomaly_logits,
        anomaly_targets,
        pos_weight=anomaly_pos_weight
    )

    total_loss = shift_loss_weight * shift_loss + anomaly_loss_weight * anomaly_loss

    return total_loss, shift_loss.detach(), anomaly_loss.detach()

In [29]:
def apply_location_predictions(shift_logits, anomaly_logits, anomaly_threshold=0.5):
    shift_pred = torch.argmax(shift_logits, dim=-1)  # (batch, length, 3)

    anomaly_prob = torch.sigmoid(anomaly_logits)

    if np.isscalar(anomaly_threshold):
        anomaly_pred = (anomaly_prob >= anomaly_threshold).long()
    else:
        anomaly_threshold = torch.tensor(
            anomaly_threshold,
            dtype=anomaly_prob.dtype,
            device=anomaly_prob.device
        ).view(1, 1, -1)

        anomaly_pred = (anomaly_prob >= anomaly_threshold).long()

    pred_loc = torch.cat([shift_pred, anomaly_pred], dim=-1)

    return pred_loc, anomaly_prob


def compute_pointwise_event_metrics_from_flat(y_true_flat, y_pred_flat):
    """
    y_true_flat and y_pred_flat shape:
        (total_time_points, 5)

    Shift columns may contain regime labels 0,1,2,...
    For pointwise event detection metrics, shifts are converted to binary:
        active shift = label > 0
    """
    y_true_binary = y_true_flat.copy()
    y_pred_binary = y_pred_flat.copy()

    # Shift columns: active if regime label > 0
    y_true_binary[:, :3] = (y_true_binary[:, :3] > 0).astype(int)
    y_pred_binary[:, :3] = (y_pred_binary[:, :3] > 0).astype(int)

    return {
        "pointwise_macro_f1": f1_score(
            y_true_binary,
            y_pred_binary,
            average="macro",
            zero_division=0
        ),
        "pointwise_micro_f1": f1_score(
            y_true_binary,
            y_pred_binary,
            average="micro",
            zero_division=0
        ),
        "pointwise_macro_precision": precision_score(
            y_true_binary,
            y_pred_binary,
            average="macro",
            zero_division=0
        ),
        "pointwise_macro_recall": recall_score(
            y_true_binary,
            y_pred_binary,
            average="macro",
            zero_division=0
        ),
        "pointwise_hamming_loss": hamming_loss(
            y_true_binary,
            y_pred_binary
        )
    }


def compute_shift_regime_accuracy_from_flat(y_true_flat, y_pred_flat):
    rows = {}

    for i, label in enumerate(EVENT_LABELS[:3]):
        rows[f"{label}_regime_accuracy"] = accuracy_score(
            y_true_flat[:, i],
            y_pred_flat[:, i]
        )

    return rows



In [12]:
def run_one_epoch_location(model, loaders, optimizer=None, anomaly_threshold=0.5):
    is_train = optimizer is not None

    if is_train:
        model.train()
        loaders = loaders.copy()
        random.shuffle(loaders)
    else:
        model.eval()

    total_loss = 0.0
    total_shift_loss = 0.0
    total_anomaly_loss = 0.0
    total_samples = 0

    all_true_flat = []
    all_pred_flat = []

    for loader in loaders:
        for X_batch, Yloc_batch in loader:
            X_batch = X_batch.to(device)
            Yloc_batch = Yloc_batch.to(device)

            if is_train:
                optimizer.zero_grad()

            with torch.set_grad_enabled(is_train):
                shift_logits, anomaly_logits = model(X_batch)

                loss, shift_loss, anomaly_loss = compute_location_loss(
                    shift_logits=shift_logits,
                    anomaly_logits=anomaly_logits,
                    Yloc=Yloc_batch,
                    anomaly_pos_weight=anomaly_pos_weight
                )

                if is_train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

            batch_size = X_batch.size(0)

            total_loss += loss.item() * batch_size
            total_shift_loss += shift_loss.item() * batch_size
            total_anomaly_loss += anomaly_loss.item() * batch_size
            total_samples += batch_size

            pred_loc, _ = apply_location_predictions(
                shift_logits,
                anomaly_logits,
                anomaly_threshold=anomaly_threshold
            )

            # Flatten each batch from (batch, length, 5) to (batch*length, 5)
            # This avoids errors when different loaders have different lengths.
            true_flat = Yloc_batch.detach().cpu().numpy().reshape(-1, len(EVENT_LABELS))
            pred_flat = pred_loc.detach().cpu().numpy().reshape(-1, len(EVENT_LABELS))

            all_true_flat.append(true_flat)
            all_pred_flat.append(pred_flat)

    y_true_flat = np.concatenate(all_true_flat, axis=0)
    y_pred_flat = np.concatenate(all_pred_flat, axis=0)

    metrics = compute_pointwise_event_metrics_from_flat(
        y_true_flat,
        y_pred_flat
    )

    metrics.update(
        compute_shift_regime_accuracy_from_flat(
            y_true_flat,
            y_pred_flat
        )
    )

    return {
        "loss": total_loss / total_samples,
        "shift_loss": total_shift_loss / total_samples,
        "anomaly_loss": total_anomaly_loss / total_samples,
        **metrics
    }

In [13]:
model = CNNBiLSTMLocationModel(
    shift_classes=SHIFT_CLASSES,
    hidden_size=64,
    dropout=0.2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 100
ANOMALY_THRESHOLD = 0.5

history = []
best_val_f1 = 0.0
best_model_path = OUTPUT_DIR / "best_location_cnn_bilstm_model.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_one_epoch_location(
        model,
        train_loaders,
        optimizer=optimizer,
        anomaly_threshold=ANOMALY_THRESHOLD
    )

    val_metrics = run_one_epoch_location(
        model,
        val_loaders,
        optimizer=None,
        anomaly_threshold=ANOMALY_THRESHOLD
    )

    row = {"epoch": epoch}

    for key, value in train_metrics.items():
        row[f"train_{key}"] = value

    for key, value in val_metrics.items():
        row[f"val_{key}"] = value

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train Pointwise Macro F1: {train_metrics['pointwise_macro_f1']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Pointwise Macro F1: {val_metrics['pointwise_macro_f1']:.4f}"
    )

    if val_metrics["pointwise_macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["pointwise_macro_f1"]

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "event_labels": EVENT_LABELS,
                "lengths": LENGTHS,
                "shift_classes": SHIFT_CLASSES,
                "epoch": epoch,
                "best_val_pointwise_macro_f1": best_val_f1,
                "anomaly_threshold": ANOMALY_THRESHOLD
            },
            best_model_path
        )

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_history_location_model.csv", index=False)

print("Training finished.")
print("Best validation pointwise macro F1:", best_val_f1)
print("Best model saved to:", best_model_path)

Epoch 01 | Train Loss: 1.0518 | Train Pointwise Macro F1: 0.0926 | Val Loss: 0.8275 | Val Pointwise Macro F1: 0.0927
Epoch 02 | Train Loss: 0.8058 | Train Pointwise Macro F1: 0.0979 | Val Loss: 0.8193 | Val Pointwise Macro F1: 0.0635
Epoch 03 | Train Loss: 0.7658 | Train Pointwise Macro F1: 0.1226 | Val Loss: 0.7609 | Val Pointwise Macro F1: 0.1283
Epoch 04 | Train Loss: 0.7236 | Train Pointwise Macro F1: 0.1620 | Val Loss: 0.7056 | Val Pointwise Macro F1: 0.1629
Epoch 05 | Train Loss: 0.6893 | Train Pointwise Macro F1: 0.1717 | Val Loss: 0.6796 | Val Pointwise Macro F1: 0.1515
Epoch 06 | Train Loss: 0.6706 | Train Pointwise Macro F1: 0.1834 | Val Loss: 0.6519 | Val Pointwise Macro F1: 0.1696
Epoch 07 | Train Loss: 0.6514 | Train Pointwise Macro F1: 0.2002 | Val Loss: 0.6698 | Val Pointwise Macro F1: 0.1615
Epoch 08 | Train Loss: 0.6554 | Train Pointwise Macro F1: 0.1906 | Val Loss: 0.6469 | Val Pointwise Macro F1: 0.1765
Epoch 09 | Train Loss: 0.6442 | Train Pointwise Macro F1: 0.2058

In [49]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

print("Loaded best model from epoch:", checkpoint["epoch"])
print("Best val pointwise macro F1:", checkpoint["best_val_pointwise_macro_f1"])

Loaded best model from epoch: 94
Best val pointwise macro F1: 0.6596337815640908


In [50]:
def extract_persistent_shift_starts(seq, min_persistence):
    """
    Extract shift start indices from a predicted or true shift-regime sequence.

    A shift start is accepted only if the active post-shift region
    continues for at least min_persistence time steps.
    """
    seq = np.asarray(seq)

    starts = []

    diff = np.diff(seq, prepend=0)
    candidate_starts = np.where(diff > 0)[0]

    for start in candidate_starts:
        new_label = seq[start]

        if new_label <= 0:
            continue

        end = start

        while end < len(seq) and seq[end] >= new_label:
            end += 1

        duration = end - start

        if duration >= min_persistence:
            starts.append(start)

    return starts


def extract_starts_from_label_sequence(seq, is_shift, min_persistence=None):
    seq = np.asarray(seq)

    if is_shift:
        if min_persistence is None:
            min_persistence = max(3, int(0.05 * len(seq)))

        starts = extract_persistent_shift_starts(
            seq,
            min_persistence=min_persistence
        )

    else:
        binary = (seq > 0).astype(int)
        diff = np.diff(binary, prepend=0)
        starts = np.where(diff == 1)[0]

    return starts


def match_start_locations(true_starts, pred_starts, tolerance):
    true_starts = list(true_starts)
    pred_starts = list(pred_starts)

    used_pred = set()
    abs_errors = []
    tp = 0

    for true_start in true_starts:
        best_j = None
        best_error = None

        for j, pred_start in enumerate(pred_starts):
            if j in used_pred:
                continue

            error = abs(pred_start - true_start)

            if best_error is None or error < best_error:
                best_error = error
                best_j = j

        if best_error is not None and best_error <= tolerance:
            tp += 1
            used_pred.add(best_j)
            abs_errors.append(best_error)

    fp = len(pred_starts) - len(used_pred)
    fn = len(true_starts) - tp

    return tp, fp, fn, abs_errors

In [ ]:
def get_location_predictions_by_length(model, loaders, lengths, anomaly_threshold=0.5):
    """
    Returns predictions separately for each sequence length.

    Output:
        predictions_by_length[length] = {
            "y_true": array with shape (num_samples, length, 5),
            "y_pred": array with shape (num_samples, length, 5)
        }
    """
    model.eval()

    predictions_by_length = {}

    with torch.no_grad():
        for length, loader in zip(lengths, loaders):
            all_true = []
            all_pred = []

            for X_batch, Yloc_batch in loader:
                X_batch = X_batch.to(device)

                shift_logits, anomaly_logits = model(X_batch)

                pred_loc, _ = apply_location_predictions(
                    shift_logits,
                    anomaly_logits,
                    anomaly_threshold=anomaly_threshold
                )
                all_true.append(Yloc_batch.cpu().numpy())
                all_pred.append(pred_loc.cpu().numpy())

            predictions_by_length[length] = {
                "y_true": np.concatenate(all_true, axis=0),
                "y_pred": np.concatenate(all_pred, axis=0)
            }

    return predictions_by_length

In [52]:
def evaluate_event_start_locations_by_length(
    predictions_by_length,
    event_labels,
    tolerance_ratio=0.05,
    shift_persistence_ratio=0.05
):
    """
    Evaluates event-start localization separately for each sequence length.

    For shifts, predicted starts are accepted only if the predicted
    post-shift region persists for at least shift_persistence_ratio
    of the sequence length.

    TN definition:
        sample-level true negative for a given event type:
        no true start exists and no predicted start exists.
    """
    rows = []

    for length, pred_dict in predictions_by_length.items():
        y_true_loc = pred_dict["y_true"]
        y_pred_loc = pred_dict["y_pred"]

        n_samples, seq_len, n_events = y_true_loc.shape

        tolerance = max(3, int(seq_len * tolerance_ratio))
        min_persistence = max(3, int(seq_len * shift_persistence_ratio))

        for event_idx, label in enumerate(event_labels):
            is_shift = event_idx < 3

            total_tp = 0
            total_fp = 0
            total_fn = 0
            total_tn = 0
            all_errors = []

            for sample_idx in range(n_samples):
                true_seq = y_true_loc[sample_idx, :, event_idx]
                pred_seq = y_pred_loc[sample_idx, :, event_idx]

                true_starts = extract_starts_from_label_sequence(
                    true_seq,
                    is_shift=is_shift,
                    min_persistence=1 if is_shift else None
)

                pred_starts = extract_starts_from_label_sequence(
                    pred_seq,
                    is_shift=is_shift,
                    min_persistence=min_persistence if is_shift else None
                )

                # Sample-level TN: no real start and no predicted start.
                if len(true_starts) == 0 and len(pred_starts) == 0:
                    total_tn += 1
                    continue

                tp, fp, fn, errors = match_start_locations(
                    true_starts=true_starts,
                    pred_starts=pred_starts,
                    tolerance=tolerance
                )

                total_tp += tp
                total_fp += fp
                total_fn += fn
                all_errors.extend(errors)

            precision = total_tp / (total_tp + total_fp + 1e-8)
            recall = total_tp / (total_tp + total_fn + 1e-8)
            f1 = 2 * precision * recall / (precision + recall + 1e-8)

            specificity = total_tn / (total_tn + total_fp + 1e-8)
            balanced_accuracy = 0.5 * (recall + specificity)

            mean_abs_error = np.mean(all_errors) if len(all_errors) > 0 else np.nan

            rows.append({
                "length": length,
                "event": label,
                "tolerance_points": tolerance,
                "shift_min_persistence": min_persistence if is_shift else np.nan,
                "tp": total_tp,
                "fp": total_fp,
                "fn": total_fn,
                "tn": total_tn,
                "start_precision": precision,
                "start_recall": recall,
                "start_f1": f1,
                "start_specificity": specificity,
                "start_balanced_accuracy": balanced_accuracy,
                "mean_abs_start_error": mean_abs_error
            })

    return pd.DataFrame(rows)

In [61]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    multilabel_confusion_matrix
)

def apply_location_predictions(shift_logits, anomaly_logits, anomaly_threshold=0.5):
    """
    Shift outputs:
        argmax over regime classes.

    Anomaly outputs:
        sigmoid + threshold.

    anomaly_threshold can be:
        - scalar, e.g. 0.5
        - array/list of length 2, e.g. [0.93, 0.95]
    """

    shift_pred = torch.argmax(shift_logits, dim=-1)

    anomaly_prob = torch.sigmoid(anomaly_logits)

    if np.isscalar(anomaly_threshold):
        anomaly_pred = (anomaly_prob >= anomaly_threshold).long()
    else:
        anomaly_threshold = torch.tensor(
            anomaly_threshold,
            dtype=anomaly_prob.dtype,
            device=anomaly_prob.device
        ).view(1, 1, -1)

        anomaly_pred = (anomaly_prob >= anomaly_threshold).long()

    pred_loc = torch.cat([shift_pred, anomaly_pred], dim=-1)

    return pred_loc, anomaly_prob


def collect_anomaly_probs_and_labels(model, loaders):
    model.eval()

    all_true = []
    all_prob = []

    with torch.no_grad():
        for loader in loaders:
            for X_batch, Yloc_batch in loader:
                X_batch = X_batch.to(device)

                _, anomaly_logits = model(X_batch)
                anomaly_prob = torch.sigmoid(anomaly_logits)

                all_true.append(
                    Yloc_batch[:, :, 3:5].cpu().numpy().reshape(-1, 2)
                )

                all_prob.append(
                    anomaly_prob.cpu().numpy().reshape(-1, 2)
                )

    y_true = np.concatenate(all_true, axis=0)
    y_prob = np.concatenate(all_prob, axis=0)

    return y_true, y_prob


def tune_anomaly_thresholds_per_class_location(
    model,
    val_loaders,
    thresholds=None
):
    """
    Tunes only anomaly thresholds using validation set.

    Tuned classes:
        point_anomaly
        collective_anomaly

    Shift classes are not threshold-tuned because their location predictions
    use argmax over shift regimes.
    """

    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.01)

    y_true, y_prob = collect_anomaly_probs_and_labels(
        model=model,
        loaders=val_loaders
    )

    rows = []
    best_thresholds = []

    for anomaly_idx, label in enumerate(EVENT_LABELS[3:5]):
        best_threshold = 0.5
        best_f1 = -1
        best_precision = 0
        best_recall = 0

        for threshold in thresholds:
            y_pred = (y_prob[:, anomaly_idx] >= threshold).astype(int)

            precision = precision_score(
                y_true[:, anomaly_idx],
                y_pred,
                zero_division=0
            )

            recall = recall_score(
                y_true[:, anomaly_idx],
                y_pred,
                zero_division=0
            )

            f1 = f1_score(
                y_true[:, anomaly_idx],
                y_pred,
                zero_division=0
            )

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
                best_precision = precision
                best_recall = recall

        best_thresholds.append(best_threshold)

        rows.append({
            "label": label,
            "best_threshold": best_threshold,
            "validation_precision": best_precision,
            "validation_recall": best_recall,
            "validation_f1": best_f1
        })

    threshold_df = pd.DataFrame(rows)

    return np.array(best_thresholds), threshold_df


TUNED_ANOMALY_THRESHOLDS, location_threshold_df = tune_anomaly_thresholds_per_class_location(
    model=model,
    val_loaders=val_loaders,
    thresholds=np.arange(0.05, 0.96, 0.01)
)

location_threshold_df.to_csv(
    OUTPUT_DIR / "validation_threshold_tuning_location_anomalies.csv",
    index=False
)

print("Tuned anomaly thresholds:", TUNED_ANOMALY_THRESHOLDS)

location_threshold_df

Tuned anomaly thresholds: [0.93 0.95]


,label,best_threshold,validation_precision,validation_recall,validation_f1
0,point_anomaly,0.93,0.401198,0.603604,0.482014
1,collective_anomaly,0.95,0.909956,0.893230,0.901515


In [62]:
location_dataset_specs = [
    ("validation", val_loaders),
    ("ood_params", ood_params_loaders),
    ("ood_background", ood_background_loaders)
]


def get_location_flat_predictions(model, loaders, anomaly_threshold=0.5):
    """
    Returns flattened pointwise true and predicted labels.

    Output:
        y_true_flat: (total_time_points, 5)
        y_pred_flat: (total_time_points, 5)
    """

    model.eval()

    all_true_flat = []
    all_pred_flat = []

    with torch.no_grad():
        for loader in loaders:
            for X_batch, Yloc_batch in loader:
                X_batch = X_batch.to(device)

                shift_logits, anomaly_logits = model(X_batch)

                pred_loc, _ = apply_location_predictions(
                    shift_logits,
                    anomaly_logits,
                    anomaly_threshold=anomaly_threshold
                )

                true_flat = Yloc_batch.cpu().numpy().reshape(-1, len(EVENT_LABELS))
                pred_flat = pred_loc.cpu().numpy().reshape(-1, len(EVENT_LABELS))

                all_true_flat.append(true_flat)
                all_pred_flat.append(pred_flat)

    y_true_flat = np.concatenate(all_true_flat, axis=0)
    y_pred_flat = np.concatenate(all_pred_flat, axis=0)

    return y_true_flat, y_pred_flat


def compute_classwise_pointwise_location_metrics(
    y_true_flat,
    y_pred_flat,
    label_names,
    dataset_name,
    setting,
    anomaly_threshold
):
    """
    Class-wise pointwise metrics.

    For shift classes:
        regime labels are converted to active/not-active using > 0.

    For anomaly classes:
        labels are already binary.
    """

    y_true_binary = y_true_flat.copy()
    y_pred_binary = y_pred_flat.copy()

    y_true_binary[:, :3] = (y_true_binary[:, :3] > 0).astype(int)
    y_pred_binary[:, :3] = (y_pred_binary[:, :3] > 0).astype(int)

    ml_cm = multilabel_confusion_matrix(y_true_binary, y_pred_binary)

    rows = []

    anomaly_threshold = np.asarray(anomaly_threshold)

    for i, label in enumerate(label_names):
        tn, fp, fn, tp = ml_cm[i].ravel()

        if i < 3:
            decision_rule = "argmax_regime"
            threshold_value = np.nan
        else:
            decision_rule = "sigmoid_threshold"

            if anomaly_threshold.ndim == 0:
                threshold_value = float(anomaly_threshold)
            else:
                threshold_value = float(anomaly_threshold[i - 3])

        rows.append({
            "dataset": dataset_name,
            "setting": setting,
            "label": label,
            "decision_rule": decision_rule,
            "threshold": threshold_value,
            "precision": precision_score(
                y_true_binary[:, i],
                y_pred_binary[:, i],
                zero_division=0
            ),
            "recall": recall_score(
                y_true_binary[:, i],
                y_pred_binary[:, i],
                zero_division=0
            ),
            "f1": f1_score(
                y_true_binary[:, i],
                y_pred_binary[:, i],
                zero_division=0
            ),
            "support": int(y_true_binary[:, i].sum()),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp)
        })

    return pd.DataFrame(rows)


def evaluate_pointwise_location_for_all_datasets(
    model,
    dataset_specs,
    anomaly_threshold,
    setting_name,
    save_suffix
):
    overall_rows = []
    classwise_rows = []

    for dataset_name, loaders in dataset_specs:
        metrics = run_one_epoch_location(
            model,
            loaders,
            optimizer=None,
            anomaly_threshold=anomaly_threshold
        )

        if np.isscalar(anomaly_threshold):
            point_threshold = float(anomaly_threshold)
            collective_threshold = float(anomaly_threshold)
        else:
            point_threshold = float(anomaly_threshold[0])
            collective_threshold = float(anomaly_threshold[1])

        overall_rows.append({
            "dataset": dataset_name,
            "setting": setting_name,
            "point_anomaly_threshold": point_threshold,
            "collective_anomaly_threshold": collective_threshold,
            **metrics
        })

        y_true_flat, y_pred_flat = get_location_flat_predictions(
            model=model,
            loaders=loaders,
            anomaly_threshold=anomaly_threshold
        )

        classwise_df = compute_classwise_pointwise_location_metrics(
            y_true_flat=y_true_flat,
            y_pred_flat=y_pred_flat,
            label_names=EVENT_LABELS,
            dataset_name=dataset_name,
            setting=setting_name,
            anomaly_threshold=anomaly_threshold
        )

        classwise_rows.append(classwise_df)

    overall_df = pd.DataFrame(overall_rows)

    classwise_df = pd.concat(
        classwise_rows,
        ignore_index=True
    )

    overall_df.to_csv(
        OUTPUT_DIR / f"location_pointwise_overall_{save_suffix}.csv",
        index=False
    )

    classwise_df.to_csv(
        OUTPUT_DIR / f"location_pointwise_classwise_{save_suffix}.csv",
        index=False
    )

    return overall_df, classwise_df


pointwise_overall_default, pointwise_classwise_default = evaluate_pointwise_location_for_all_datasets(
    model=model,
    dataset_specs=location_dataset_specs,
    anomaly_threshold=0.5,
    setting_name="default_0.5",
    save_suffix="default_threshold"
)

pointwise_overall_tuned, pointwise_classwise_tuned = evaluate_pointwise_location_for_all_datasets(
    model=model,
    dataset_specs=location_dataset_specs,
    anomaly_threshold=TUNED_ANOMALY_THRESHOLDS,
    setting_name="tuned",
    save_suffix="tuned_threshold"
)

display(pointwise_overall_default)
display(pointwise_classwise_default)

display(pointwise_overall_tuned)
display(pointwise_classwise_tuned)

,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold,loss,shift_loss,anomaly_loss,pointwise_macro_f1,pointwise_micro_f1,pointwise_macro_precision,pointwise_macro_recall,pointwise_hamming_loss,mean_shift_regime_accuracy,variance_shift_regime_accuracy,trend_shift_regime_accuracy
0,validation,default_0.5,0.5,0.5,0.536289,0.327265,0.209025,0.659634,0.727016,0.686811,0.731844,0.054859,0.905661,0.900818,0.874568
1,ood_params,default_0.5,0.5,0.5,0.687594,0.486301,0.201293,0.524997,0.562707,0.602152,0.484403,0.082107,0.869648,0.859674,0.819967
2,ood_background,default_0.5,0.5,0.5,1.141484,0.590157,0.551326,0.493845,0.471954,0.497177,0.601672,0.112858,0.745089,0.829179,0.826295


,dataset,setting,label,decision_rule,threshold,precision,recall,f1,support,tn,fp,fn,tp
0,validation,default_0.5,mean_shift,argmax_regime,NaN,0.820204,0.759367,0.788614,103394,455395,17211,24880,78514
1,validation,default_0.5,variance_shift,argmax_regime,NaN,0.836405,0.679829,0.750032,102820,459508,13672,32920,69900
2,validation,default_0.5,trend_shift,argmax_regime,NaN,0.780183,0.476065,0.591313,97680,465218,13102,51178,46502
3,validation,default_0.5,point_anomaly,sigmoid_threshold,0.5,0.182927,0.810811,0.298507,111,575487,402,21,90
4,validation,default_0.5,collective_anomaly,sigmoid_threshold,0.5,0.814336,0.933147,0.869703,16484,556009,3507,1102,15382
5,ood_params,default_0.5,mean_shift,argmax_regime,NaN,0.720739,0.563769,0.632663,96951,457871,21178,42293,54658
6,ood_params,default_0.5,variance_shift,argmax_regime,NaN,0.741514,0.437301,0.550154,101358,459191,15451,57034,44324
7,ood_params,default_0.5,trend_shift,argmax_regime,NaN,0.558076,0.318140,0.405257,104146,445617,26237,71013,33133
8,ood_params,default_0.5,point_anomaly,sigmoid_threshold,0.5,0.083832,0.148936,0.107280,94,575753,153,80,14
9,ood_params,default_0.5,collective_anomaly,sigmoid_threshold,0.5,0.906600,0.953867,0.929633,20983,552955,2062,968,20015


,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold,loss,shift_loss,anomaly_loss,pointwise_macro_f1,pointwise_micro_f1,pointwise_macro_precision,pointwise_macro_recall,pointwise_hamming_loss,mean_shift_regime_accuracy,variance_shift_regime_accuracy,trend_shift_regime_accuracy
0,validation,tuned,0.93,0.95,0.536289,0.327265,0.209025,0.702698,0.728481,0.749589,0.682419,0.054279,0.905661,0.900818,0.874568
1,ood_params,tuned,0.93,0.95,0.687594,0.486301,0.201293,0.520257,0.561049,0.634124,0.453282,0.081975,0.869648,0.859674,0.819967
2,ood_background,tuned,0.93,0.95,1.141484,0.590157,0.551326,0.520532,0.471214,0.544583,0.546112,0.112384,0.745089,0.829179,0.826295


,dataset,setting,label,decision_rule,threshold,precision,recall,f1,support,tn,fp,fn,tp
0,validation,tuned,mean_shift,argmax_regime,NaN,0.820204,0.759367,0.788614,103394,455395,17211,24880,78514
1,validation,tuned,variance_shift,argmax_regime,NaN,0.836405,0.679829,0.750032,102820,459508,13672,32920,69900
2,validation,tuned,trend_shift,argmax_regime,NaN,0.780183,0.476065,0.591313,97680,465218,13102,51178,46502
3,validation,tuned,point_anomaly,sigmoid_threshold,0.93,0.401198,0.603604,0.482014,111,575789,100,44,67
4,validation,tuned,collective_anomaly,sigmoid_threshold,0.95,0.909956,0.893230,0.901515,16484,558059,1457,1760,14724
5,ood_params,tuned,mean_shift,argmax_regime,NaN,0.720739,0.563769,0.632663,96951,457871,21178,42293,54658
6,ood_params,tuned,variance_shift,argmax_regime,NaN,0.741514,0.437301,0.550154,101358,459191,15451,57034,44324
7,ood_params,tuned,trend_shift,argmax_regime,NaN,0.558076,0.318140,0.405257,104146,445617,26237,71013,33133
8,ood_params,tuned,point_anomaly,sigmoid_threshold,0.93,0.178571,0.053191,0.081967,94,575883,23,89,5
9,ood_params,tuned,collective_anomaly,sigmoid_threshold,0.95,0.971717,0.894009,0.931245,20983,554471,546,2224,18759


In [63]:
def evaluate_start_locations_for_all_datasets(
    model,
    dataset_specs,
    lengths,
    anomaly_threshold,
    setting_name,
    tolerance_ratio=0.05,
    shift_persistence_ratio=0.1,
    save_suffix="default_threshold"
):
    start_classwise_rows = []

    for dataset_name, loaders in dataset_specs:
        predictions_by_length = get_location_predictions_by_length(
            model=model,
            loaders=loaders,
            lengths=lengths,
            anomaly_threshold=anomaly_threshold
        )

        start_df = evaluate_event_start_locations_by_length(
            predictions_by_length=predictions_by_length,
            event_labels=EVENT_LABELS,
            tolerance_ratio=tolerance_ratio,
            shift_persistence_ratio=shift_persistence_ratio
        )

        start_df["dataset"] = dataset_name
        start_df["setting"] = setting_name

        if np.isscalar(anomaly_threshold):
            start_df["point_anomaly_threshold"] = float(anomaly_threshold)
            start_df["collective_anomaly_threshold"] = float(anomaly_threshold)
        else:
            start_df["point_anomaly_threshold"] = float(anomaly_threshold[0])
            start_df["collective_anomaly_threshold"] = float(anomaly_threshold[1])

        start_classwise_rows.append(start_df)

    start_classwise_df = pd.concat(
        start_classwise_rows,
        ignore_index=True
    )

    overall_rows = []

    for (dataset_name, setting_name), group in start_classwise_df.groupby(["dataset", "setting"]):
        total_tp = group["tp"].sum()
        total_fp = group["fp"].sum()
        total_fn = group["fn"].sum()
        total_tn = group["tn"].sum()

        start_precision = total_tp / (total_tp + total_fp + 1e-8)
        start_recall = total_tp / (total_tp + total_fn + 1e-8)
        start_f1 = 2 * start_precision * start_recall / (start_precision + start_recall + 1e-8)

        start_specificity = total_tn / (total_tn + total_fp + 1e-8)
        start_balanced_accuracy = 0.5 * (start_recall + start_specificity)

        valid_error_rows = group.dropna(subset=["mean_abs_start_error"])

        if len(valid_error_rows) > 0:
            weighted_error = np.average(
                valid_error_rows["mean_abs_start_error"],
                weights=valid_error_rows["tp"].clip(lower=1)
            )
        else:
            weighted_error = np.nan

        overall_rows.append({
            "dataset": dataset_name,
            "setting": setting_name,
            "point_anomaly_threshold": group["point_anomaly_threshold"].iloc[0],
            "collective_anomaly_threshold": group["collective_anomaly_threshold"].iloc[0],
            "tp": total_tp,
            "fp": total_fp,
            "fn": total_fn,
            "tn": total_tn,
            "start_precision": start_precision,
            "start_recall": start_recall,
            "start_f1": start_f1,
            "start_specificity": start_specificity,
            "start_balanced_accuracy": start_balanced_accuracy,
            "mean_abs_start_error": weighted_error
        })

    start_overall_df = pd.DataFrame(overall_rows)

    start_overall_df.to_csv(
        OUTPUT_DIR / f"location_start_overall_{save_suffix}.csv",
        index=False
    )

    start_classwise_df.to_csv(
        OUTPUT_DIR / f"location_start_classwise_by_length_{save_suffix}.csv",
        index=False
    )

    return start_overall_df, start_classwise_df


start_overall_default, start_classwise_default = evaluate_start_locations_for_all_datasets(
    model=model,
    dataset_specs=location_dataset_specs,
    lengths=LENGTHS,
    anomaly_threshold=0.5,
    setting_name="default_0.5",
    tolerance_ratio=0.05,
    shift_persistence_ratio=0.1,
    save_suffix="default_threshold"
)

start_overall_tuned, start_classwise_tuned = evaluate_start_locations_for_all_datasets(
    model=model,
    dataset_specs=location_dataset_specs,
    lengths=LENGTHS,
    anomaly_threshold=TUNED_ANOMALY_THRESHOLDS,
    setting_name="tuned",
    tolerance_ratio=0.05,
    shift_persistence_ratio=0.1,
    save_suffix="tuned_threshold"
)

display(start_overall_default)
display(start_classwise_default)

display(start_overall_tuned)
display(start_classwise_tuned)

,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold,tp,fp,fn,tn,start_precision,start_recall,start_f1,start_specificity,start_balanced_accuracy,mean_abs_start_error
0,ood_background,default_0.5,0.5,0.5,413,1228,764,2043,0.251676,0.350892,0.293116,0.624580,0.487736,5.738499
1,ood_params,default_0.5,0.5,0.5,453,683,757,2273,0.398768,0.374380,0.386189,0.768945,0.571662,6.693157
2,validation,default_0.5,0.5,0.5,626,924,559,2233,0.403871,0.528270,0.457770,0.707317,0.617794,6.781150


,length,event,tolerance_points,shift_min_persistence,tp,fp,fn,tn,start_precision,start_recall,start_f1,start_specificity,start_balanced_accuracy,mean_abs_start_error,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold
0,400,mean_shift,20,40.0,46,38,50,147,0.547619,0.479167,0.511111,0.794595,0.636881,6.260870,validation,default_0.5,0.5,0.5
1,400,variance_shift,20,40.0,35,38,65,155,0.479452,0.350000,0.404624,0.803109,0.576554,7.314286,validation,default_0.5,0.5,0.5
2,400,trend_shift,20,40.0,19,38,73,155,0.333333,0.206522,0.255034,0.803109,0.504815,8.842105,validation,default_0.5,0.5,0.5
3,400,point_anomaly,20,NaN,26,79,9,156,0.247619,0.742857,0.371429,0.663830,0.703343,0.000000,validation,default_0.5,0.5,0.5
4,400,collective_anomaly,20,NaN,64,28,11,150,0.695652,0.853333,0.766467,0.842697,0.848015,2.359375,validation,default_0.5,0.5,0.5
5,800,mean_shift,40,80.0,51,32,40,154,0.614458,0.560440,0.586207,0.827957,0.694198,5.941176,validation,default_0.5,0.5,0.5
6,800,variance_shift,40,80.0,43,38,52,156,0.530864,0.452632,0.488636,0.804124,0.628378,13.697674,validation,default_0.5,0.5,0.5
7,800,trend_shift,40,80.0,15,41,77,156,0.267857,0.163043,0.202703,0.791878,0.477461,19.933333,validation,default_0.5,0.5,0.5
8,800,point_anomaly,40,NaN,30,122,6,153,0.197368,0.833333,0.319149,0.556364,0.694848,0.833333,validation,default_0.5,0.5,0.5
9,800,collective_anomaly,40,NaN,72,50,3,139,0.590164,0.960000,0.730964,0.735450,0.847725,2.763889,validation,default_0.5,0.5,0.5


,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold,tp,fp,fn,tn,start_precision,start_recall,start_f1,start_specificity,start_balanced_accuracy,mean_abs_start_error
0,ood_background,tuned,0.93,0.95,378,789,799,2207,0.323907,0.321155,0.322526,0.736649,0.528902,6.166667
1,ood_params,tuned,0.93,0.95,427,426,783,2435,0.500586,0.352893,0.413960,0.851101,0.601997,6.889930
2,validation,tuned,0.93,0.95,598,477,587,2417,0.556279,0.504641,0.529204,0.835176,0.669909,6.887960


,length,event,tolerance_points,shift_min_persistence,tp,fp,fn,tn,start_precision,start_recall,start_f1,start_specificity,start_balanced_accuracy,mean_abs_start_error,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold
0,400,mean_shift,20,40.0,46,38,50,147,0.547619,0.479167,0.511111,0.794595,0.636881,6.260870,validation,tuned,0.93,0.95
1,400,variance_shift,20,40.0,35,38,65,155,0.479452,0.350000,0.404624,0.803109,0.576554,7.314286,validation,tuned,0.93,0.95
2,400,trend_shift,20,40.0,19,38,73,155,0.333333,0.206522,0.255034,0.803109,0.504815,8.842105,validation,tuned,0.93,0.95
3,400,point_anomaly,20,NaN,16,27,19,183,0.372093,0.457143,0.410256,0.871429,0.664286,0.000000,validation,tuned,0.93,0.95
4,400,collective_anomaly,20,NaN,62,13,13,162,0.826667,0.826667,0.826667,0.925714,0.876190,1.951613,validation,tuned,0.93,0.95
5,800,mean_shift,40,80.0,51,32,40,154,0.614458,0.560440,0.586207,0.827957,0.694198,5.941176,validation,tuned,0.93,0.95
6,800,variance_shift,40,80.0,43,38,52,156,0.530864,0.452632,0.488636,0.804124,0.628378,13.697674,validation,tuned,0.93,0.95
7,800,trend_shift,40,80.0,15,41,77,156,0.267857,0.163043,0.202703,0.791878,0.477461,19.933333,validation,tuned,0.93,0.95
8,800,point_anomaly,40,NaN,21,28,15,189,0.428571,0.583333,0.494118,0.870968,0.727151,0.000000,validation,tuned,0.93,0.95
9,800,collective_anomaly,40,NaN,71,11,4,162,0.865854,0.946667,0.904459,0.936416,0.941541,2.366197,validation,tuned,0.93,0.95


In [64]:
def make_wide_comparison(default_df, tuned_df, merge_keys, output_name):
    comparison_wide = default_df.merge(
        tuned_df,
        on=merge_keys,
        suffixes=("_default", "_tuned")
    )

    for col in default_df.columns:
        if col in merge_keys:
            continue

        if pd.api.types.is_numeric_dtype(default_df[col]):
            comparison_wide[f"{col}_change"] = (
                comparison_wide[f"{col}_tuned"] -
                comparison_wide[f"{col}_default"]
            )

    comparison_wide.to_csv(
        OUTPUT_DIR / output_name,
        index=False
    )

    return comparison_wide


# A1. Overall pointwise comparison
pointwise_overall_comparison = make_wide_comparison(
    default_df=pointwise_overall_default,
    tuned_df=pointwise_overall_tuned,
    merge_keys=["dataset"],
    output_name="comparison_pointwise_overall_default_vs_tuned.csv"
)

# A2. Class-wise pointwise comparison
pointwise_classwise_comparison = make_wide_comparison(
    default_df=pointwise_classwise_default,
    tuned_df=pointwise_classwise_tuned,
    merge_keys=["dataset", "label"],
    output_name="comparison_pointwise_classwise_default_vs_tuned.csv"
)

# B1. Overall event-start comparison
start_overall_comparison = make_wide_comparison(
    default_df=start_overall_default,
    tuned_df=start_overall_tuned,
    merge_keys=["dataset"],
    output_name="comparison_start_overall_default_vs_tuned.csv"
)

# B2. Class-wise/event-wise event-start comparison
start_classwise_comparison = make_wide_comparison(
    default_df=start_classwise_default,
    tuned_df=start_classwise_tuned,
    merge_keys=["dataset", "length", "event"],
    output_name="comparison_start_classwise_by_length_default_vs_tuned.csv"
)

display(pointwise_overall_comparison)
display(pointwise_classwise_comparison)
display(start_overall_comparison)
display(start_classwise_comparison)

,dataset,setting_default,point_anomaly_threshold_default,collective_anomaly_threshold_default,loss_default,shift_loss_default,anomaly_loss_default,pointwise_macro_f1_default,pointwise_micro_f1_default,pointwise_macro_precision_default,...,shift_loss_change,anomaly_loss_change,pointwise_macro_f1_change,pointwise_micro_f1_change,pointwise_macro_precision_change,pointwise_macro_recall_change,pointwise_hamming_loss_change,mean_shift_regime_accuracy_change,variance_shift_regime_accuracy_change,trend_shift_regime_accuracy_change
0,validation,default_0.5,0.5,0.5,0.536289,0.327265,0.209025,0.659634,0.727016,0.686811,...,0.0,0.0,0.043064,0.001464,0.062778,-0.049425,-0.000580,0.0,0.0,0.0
1,ood_params,default_0.5,0.5,0.5,0.687594,0.486301,0.201293,0.524997,0.562707,0.602152,...,0.0,0.0,-0.004740,-0.001658,0.031971,-0.031121,-0.000132,0.0,0.0,0.0
2,ood_background,default_0.5,0.5,0.5,1.141484,0.590157,0.551326,0.493845,0.471954,0.497177,...,0.0,0.0,0.026687,-0.000740,0.047406,-0.055559,-0.000474,0.0,0.0,0.0


,dataset,setting_default,label,decision_rule_default,threshold_default,precision_default,recall_default,f1_default,support_default,tn_default,...,tp_tuned,threshold_change,precision_change,recall_change,f1_change,support_change,tn_change,fp_change,fn_change,tp_change
0,validation,default_0.5,mean_shift,argmax_regime,NaN,0.820204,0.759367,0.788614,103394,455395,...,78514,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
1,validation,default_0.5,variance_shift,argmax_regime,NaN,0.836405,0.679829,0.750032,102820,459508,...,69900,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
2,validation,default_0.5,trend_shift,argmax_regime,NaN,0.780183,0.476065,0.591313,97680,465218,...,46502,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
3,validation,default_0.5,point_anomaly,sigmoid_threshold,0.5,0.182927,0.810811,0.298507,111,575487,...,67,0.43,0.218271,-0.207207,0.183507,0,302,-302,23,-23
4,validation,default_0.5,collective_anomaly,sigmoid_threshold,0.5,0.814336,0.933147,0.869703,16484,556009,...,14724,0.45,0.095620,-0.039917,0.031813,0,2050,-2050,658,-658
5,ood_params,default_0.5,mean_shift,argmax_regime,NaN,0.720739,0.563769,0.632663,96951,457871,...,54658,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
6,ood_params,default_0.5,variance_shift,argmax_regime,NaN,0.741514,0.437301,0.550154,101358,459191,...,44324,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
7,ood_params,default_0.5,trend_shift,argmax_regime,NaN,0.558076,0.318140,0.405257,104146,445617,...,33133,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
8,ood_params,default_0.5,point_anomaly,sigmoid_threshold,0.5,0.083832,0.148936,0.107280,94,575753,...,5,0.43,0.094739,-0.095745,-0.025312,0,130,-130,9,-9
9,ood_params,default_0.5,collective_anomaly,sigmoid_threshold,0.5,0.906600,0.953867,0.929633,20983,552955,...,18759,0.45,0.065118,-0.059858,0.001612,0,1516,-1516,1256,-1256


,dataset,setting_default,point_anomaly_threshold_default,collective_anomaly_threshold_default,tp_default,fp_default,fn_default,tn_default,start_precision_default,start_recall_default,...,tp_change,fp_change,fn_change,tn_change,start_precision_change,start_recall_change,start_f1_change,start_specificity_change,start_balanced_accuracy_change,mean_abs_start_error_change
0,ood_background,default_0.5,0.5,0.5,413,1228,764,2043,0.251676,0.350892,...,-35,-439,35,164,0.072232,-0.029737,0.029410,0.112069,0.041166,0.428168
1,ood_params,default_0.5,0.5,0.5,453,683,757,2273,0.398768,0.374380,...,-26,-257,26,162,0.101819,-0.021488,0.027771,0.082156,0.030334,0.196773
2,validation,default_0.5,0.5,0.5,626,924,559,2233,0.403871,0.528270,...,-28,-447,28,184,0.152408,-0.023629,0.071434,0.127859,0.052115,0.106810


,length,event,tolerance_points_default,shift_min_persistence_default,tp_default,fp_default,fn_default,tn_default,start_precision_default,start_recall_default,...,fn_change,tn_change,start_precision_change,start_recall_change,start_f1_change,start_specificity_change,start_balanced_accuracy_change,mean_abs_start_error_change,point_anomaly_threshold_change,collective_anomaly_threshold_change
0,400,mean_shift,20,40.0,46,38,50,147,0.547619,0.479167,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.43,0.45
1,400,variance_shift,20,40.0,35,38,65,155,0.479452,0.350000,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.43,0.45
2,400,trend_shift,20,40.0,19,38,73,155,0.333333,0.206522,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.43,0.45
3,400,point_anomaly,20,NaN,26,79,9,156,0.247619,0.742857,...,10,27,0.124474,-0.285714,0.038828,0.207599,-0.039058,0.000000,0.43,0.45
4,400,collective_anomaly,20,NaN,64,28,11,150,0.695652,0.853333,...,2,12,0.131014,-0.026667,0.060200,0.083018,0.028175,-0.407762,0.43,0.45
5,800,mean_shift,40,80.0,51,32,40,154,0.614458,0.560440,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.43,0.45
6,800,variance_shift,40,80.0,43,38,52,156,0.530864,0.452632,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.43,0.45
7,800,trend_shift,40,80.0,15,41,77,156,0.267857,0.163043,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.43,0.45
8,800,point_anomaly,40,NaN,30,122,6,153,0.197368,0.833333,...,9,36,0.231203,-0.250000,0.174969,0.314604,0.032302,-0.833333,0.43,0.45
9,800,collective_anomaly,40,NaN,72,50,3,139,0.590164,0.960000,...,1,23,0.275690,-0.013333,0.173494,0.200966,0.093817,-0.397692,0.43,0.45
